# Task 2 — Dimensionality Reduction

Compare two feature reduction strategies (TF-IDF feature selection vs TruncatedSVD/PCA)
at 4 sizes (2000, 1000, 500, 100), evaluated with KNN (n_neighbors=2).

Generates 8 Kaggle submission CSVs.

In [2]:
# Section 0 — Imports & Config
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    'N_FEATURES': [2000, 1000, 500, 100],
    'N_NEIGHBORS': 2,
    'TFIDF_BASE_FEATURES': 5000,
    'TRAIN_PATH': 'dataset/train.csv',
    'TEST_PATH':  'dataset/test.csv',
    'SAMPLE_PATH': 'dataset/sample_submission_random_guess.csv',
    'SUBMISSIONS_DIR': 'submissions',
}
os.makedirs(CONFIG['SUBMISSIONS_DIR'], exist_ok=True)
print('Config:', CONFIG)

Config: {'N_FEATURES': [2000, 1000, 500, 100], 'N_NEIGHBORS': 2, 'TFIDF_BASE_FEATURES': 5000, 'TRAIN_PATH': 'dataset/train.csv', 'TEST_PATH': 'dataset/test.csv', 'SAMPLE_PATH': 'dataset/sample_submission_random_guess.csv', 'SUBMISSIONS_DIR': 'submissions'}


## Section 1 — Load Data

In [6]:
# Section 1 — Load Data
train_df  = pd.read_csv(CONFIG['TRAIN_PATH'], sep='\t')
test_df   = pd.read_csv(CONFIG['TEST_PATH'],  sep='\t')
sample_df = pd.read_csv(CONFIG['SAMPLE_PATH'])

train_texts  = train_df['abstract'].astype(str).tolist()
train_labels = train_df['label_id'].tolist()
test_texts   = test_df['abstract'].astype(str).tolist()
test_ids     = test_df['id'].tolist()

print(f'Train: {len(train_texts)} samples')
print(f'Test:  {len(test_texts)} samples')
print(f'Classes: {len(set(train_labels))}')
train_df.head(3)

Train: 139156 samples
Test:  34790 samples
Classes: 39


,id,abstract,label_id
0,95829,Project-based learning plays a crucial role in...,14
1,73195,Edge computing is a distributed computing pa...,10
2,22319,"In today's computing environment, where Arti...",3


## Section 2 — Baseline TF-IDF (5000 features)

Fit once on train; used as input for Method B (TruncatedSVD).

In [7]:
# Section 2 — Baseline TF-IDF (5000 features)
base_vectorizer = TfidfVectorizer(max_features=CONFIG['TFIDF_BASE_FEATURES'])
X_train_base = base_vectorizer.fit_transform(train_texts)
X_test_base  = base_vectorizer.transform(test_texts)

print(f'X_train_base shape: {X_train_base.shape}')
print(f'X_test_base shape:  {X_test_base.shape}')

X_train_base shape: (139156, 5000)
X_test_base shape:  (34790, 5000)


## Section 3 — Method A: Feature Selection (TF-IDF restriction)

For each N, re-fit `TfidfVectorizer(max_features=N)` from scratch on train text.

In [8]:
# Section 3 — Method A: TF-IDF Feature Selection
knn = KNeighborsClassifier(n_neighbors=CONFIG['N_NEIGHBORS'], metric='cosine', n_jobs=-1)

for N in CONFIG['N_FEATURES']:
    print(f'\n--- TF-IDF, N={N} ---')
    vec = TfidfVectorizer(max_features=N)
    X_tr = vec.fit_transform(train_texts)
    X_te = vec.transform(test_texts)
    print(f'  Train shape: {X_tr.shape}')

    knn.fit(X_tr, train_labels)
    preds = knn.predict(X_te)

    out = pd.DataFrame({'id': test_ids, 'label_id': preds})
    path = os.path.join(CONFIG['SUBMISSIONS_DIR'], f'task2_tfidf_{N}.csv')
    out.to_csv(path, index=False)
    print(f'  Saved: {path}')

print('\nMethod A complete.')


--- TF-IDF, N=2000 ---
  Train shape: (139156, 2000)


Python(19092) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Saved: submissions/task2_tfidf_2000.csv

--- TF-IDF, N=1000 ---
  Train shape: (139156, 1000)
  Saved: submissions/task2_tfidf_1000.csv

--- TF-IDF, N=500 ---
  Train shape: (139156, 500)
  Saved: submissions/task2_tfidf_500.csv

--- TF-IDF, N=100 ---
  Train shape: (139156, 100)
  Saved: submissions/task2_tfidf_100.csv

Method A complete.


## Section 4 — Method B: PCA via TruncatedSVD

Start from the 5000-feature TF-IDF matrix and reduce with `TruncatedSVD(n_components=N)`.

In [9]:
# Section 4 — Method B: TruncatedSVD (PCA equivalent for sparse matrices)
knn = KNeighborsClassifier(n_neighbors=CONFIG['N_NEIGHBORS'], metric='euclidean', n_jobs=-1)

for N in CONFIG['N_FEATURES']:
    print(f'\n--- TruncatedSVD, N={N} ---')
    svd = TruncatedSVD(n_components=N, random_state=42)
    X_tr = svd.fit_transform(X_train_base)
    X_te = svd.transform(X_test_base)
    explained = svd.explained_variance_ratio_.sum()
    print(f'  Train shape: {X_tr.shape}  |  Explained variance: {explained:.3f}')

    knn.fit(X_tr, train_labels)
    preds = knn.predict(X_te)

    out = pd.DataFrame({'id': test_ids, 'label_id': preds})
    path = os.path.join(CONFIG['SUBMISSIONS_DIR'], f'task2_pca_{N}.csv')
    out.to_csv(path, index=False)
    print(f'  Saved: {path}')

print('\nMethod B complete.')


--- TruncatedSVD, N=2000 ---
  Train shape: (139156, 2000)  |  Explained variance: 0.741


Python(34990) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Saved: submissions/task2_pca_2000.csv

--- TruncatedSVD, N=1000 ---
  Train shape: (139156, 1000)  |  Explained variance: 0.540
  Saved: submissions/task2_pca_1000.csv

--- TruncatedSVD, N=500 ---
  Train shape: (139156, 500)  |  Explained variance: 0.381
  Saved: submissions/task2_pca_500.csv

--- TruncatedSVD, N=100 ---
  Train shape: (139156, 100)  |  Explained variance: 0.157
  Saved: submissions/task2_pca_100.csv

Method B complete.


## Section 5 — Results Table

After submitting the 8 CSVs to Kaggle, fill in the Macro F1 scores below.

In [3]:
# Section 5 — Results Table
# Kaggle Macro F1 scores (public leaderboard)
results = {
    'N': [2000, 1000, 500, 100],
    'TF-IDF Selection (Macro F1)':   [0.398, 0.352, 0.298, 0.117],
    'TruncatedSVD / PCA (Macro F1)': [0.337, 0.364, 0.396, 0.387],
}
results_df = pd.DataFrame(results).set_index('N')
print(results_df)

      TF-IDF Selection (Macro F1)  TruncatedSVD / PCA (Macro F1)
N                                                               
2000                        0.398                          0.337
1000                        0.352                          0.364
500                         0.298                          0.396
100                         0.117                          0.387


### Analysis

- **Best overall setting:** TF-IDF feature selection with `N=2000` gives the highest Kaggle Macro F1 (`0.398`).
- **TF-IDF trend:** Performance drops as `N` decreases (`0.398 -> 0.352 -> 0.298 -> 0.117`), suggesting aggressive feature truncation removes important discriminative words.
- **PCA trend:** PCA is more stable across smaller dimensions (`0.337, 0.364, 0.396, 0.387`), and outperforms TF-IDF at `N=500` and `N=100`.
- **Method comparison:** TF-IDF is strongest at high dimensionality (`N=2000`), while PCA is more robust when the representation is compressed.